# Simple CNN on MNIST – Student Version

In this notebook we train a **Convolutional Neural Network (CNN)** on the
classic **MNIST handwritten digits** dataset.

By the end of this notebook, you should be able to:

- Load and inspect the MNIST dataset
- Prepare image data for a CNN (reshape + normalize)
- Build a small CNN using Keras
- Understand the main CNN parameters (filters, kernel size, activation, etc.)
- Train, evaluate, and interpret the model's performance
- Visualize a few predictions


## 1. Import libraries

Here we import the libraries that we will use:

- **NumPy**: numerical operations on arrays.
- **TensorFlow / Keras**: building and training neural networks.
- **MNIST**: built-in dataset of handwritten digits (0–9).
- **Matplotlib**: plotting images and results.


In [ ]:
# Import standard libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
import matplotlib.pyplot as plt

# Always useful to check the TensorFlow version
print("TensorFlow version:", tf.__version__)

## 2. Load the MNIST dataset

MNIST is a dataset of **28×28 grayscale images** of handwritten digits.
It is split into:

- 60,000 training images (`x_train`) with labels (`y_train`)
- 10,000 test images (`x_test`) with labels (`y_test`)

Each label is an integer from **0 to 9** indicating which digit is in the image.


In [ ]:
# Load data: Keras returns train and test splits for us.
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("Train images shape:", x_train.shape)   # (60000, 28, 28)
print("Train labels shape:", y_train.shape)   # (60000,)
print("Test images shape:", x_test.shape)     # (10000, 28, 28)
print("Test labels shape:", y_test.shape)     # (10000,)

### Quick look at some sample digits

Let's display a few **raw images** before any preprocessing.
This helps us understand what the network will see.


In [ ]:
# Plot the first 5 training images with their labels
plt.figure(figsize=(8, 2))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")  # grayscale image
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.suptitle("Example raw MNIST digits")
plt.tight_layout()
plt.show()

## 3. Preprocess the data

CNNs expect inputs as **tensors** with shape `(height, width, channels)`.
MNIST images are 28×28 and **grayscale**, so the number of channels is 1.

We also **normalize** pixel values from `[0, 255]` to `[0, 1]` to help the network train
more stably and faster.


In [ ]:
# Convert pixel values from integers 0–255 to floats 0–1
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

# Add a channel dimension at the end: (28, 28) -> (28, 28, 1)
x_train = np.expand_dims(x_train, axis=-1)
x_test  = np.expand_dims(x_test, axis=-1)

print("New train shape:", x_train.shape)  # (60000, 28, 28, 1)
print("New test shape:", x_test.shape)    # (10000, 28, 28, 1)

## 4. Build a simple CNN model

Now we define a **Convolutional Neural Network**. The main layers are:

- `Conv2D(filters, kernel_size, activation)`: learns small filters that slide over the image.
  - **filters**: number of feature maps (e.g., 32, 64). More filters can learn more patterns.
  - **kernel_size**: size of the filter (e.g., 3×3). Small kernels work well for images.
  - **activation**: non-linear function. We use **ReLU** (Rectified Linear Unit).
- `MaxPooling2D()`: down-samples the feature maps, keeping the most important activations.
- `Flatten()`: converts 2D feature maps to a 1D vector.
- `Dense(units, activation)`: fully connected layer.
- Final `Dense(10, softmax)`: outputs probabilities for each of the 10 digit classes.


In [ ]:
# Define a small CNN architecture
model = models.Sequential([
    # Input layer expects images of shape 28x28 with 1 channel
    layers.Input(shape=(28, 28, 1)),

    # First convolutional layer
    # 32 filters, each of size 3x3, using ReLU activation
    layers.Conv2D(32, kernel_size=3, activation="relu"),
    # Reduces spatial size (height and width) by taking max over 2x2 blocks
    layers.MaxPooling2D(pool_size=2),

    # Second convolutional layer with more filters (64)
    layers.Conv2D(64, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    # Flatten the 2D feature maps into a 1D vector
    layers.Flatten(),

    # Fully connected (Dense) layer with 64 neurons
    layers.Dense(64, activation="relu"),

    # Output layer: 10 neurons for 10 digit classes (0–9), softmax gives probabilities
    layers.Dense(10, activation="softmax")
])

# Compile the model: choose optimizer, loss function, and metrics
# - optimizer='adam': adaptive learning rate, works well in many cases
# - loss='sparse_categorical_crossentropy': for integer labels (0–9)
# - metrics=['accuracy']: we monitor classification accuracy
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Show a summary of the model architecture
model.summary()

## 5. Train the model

We now train the CNN on the training data.

Important parameters:

- **epochs**: how many times we iterate over the whole training set.
- **batch_size**: how many samples we use for one gradient update.
- **validation_split**: fraction of the training data used to monitor validation performance.


In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=5,          # try increasing to 10 for better accuracy
    batch_size=64,     # 64 images per mini-batch
    validation_split=0.1,  # 10% of training data used for validation
    verbose=2          # prints one line per epoch
)

## 6. Evaluate on test data

We now evaluate the trained model on the **test set**, which the model has never seen during training.
This gives us a more realistic measure of generalization performance.


In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.3f}")
print(f"Test loss: {test_loss:.3f}")

## 7. Visualize some predictions

Finally, we pick a few random test images, let the model predict the digit for each,
and compare the **true label** with the **predicted label**.


In [ ]:
def show_predictions(n=5):
    """Show n random test images with true and predicted labels."""
    # Randomly select n indices from the test set
    indices = np.random.choice(len(x_test), size=n, replace=False)
    images = x_test[indices]
    labels = y_test[indices]

    # Model outputs probabilities for each class (0–9)
    preds = model.predict(images)
    pred_labels = np.argmax(preds, axis=1)

    plt.figure(figsize=(10, 2))
    for i in range(n):
        plt.subplot(1, n, i + 1)
        plt.imshow(images[i].squeeze(), cmap="gray")
        plt.title(f"True: {labels[i]}\nPred: {pred_labels[i]}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# Show 5 example predictions
show_predictions(5)